# Example Notebook 03:

Model Training of base TimeXer learner with MLFlow integration.

In [ ]:
%load_ext autoreload
%autoreload 2

from lightning import Trainer
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
import mlflow

from src import BESSTimeXer, TimeXerDataModule
import config

In [ ]:
# Prepare data.
data = TimeXerDataModule(**config.DATA_DE_CONFIG)

In [ ]:
# Set tracking URI directly to local MLFlow database.
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
# ... or, set tracking URI to default MLFlow local port when using MLFlow UI. Run `mlflow ui` in terminal to initialize MLFlow UI.
# mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXPERIMENT_NAME = "BESSTimeXer"
CONFIG = config.MODEL_CONFIG | config.DATA_DE_CONFIG

In [ ]:
# Set up logging with MLFlow.
mlflow_logger = MLFlowLogger(
    experiment_name=EXPERIMENT_NAME,
    run_name=config.MODEL_CONFIG['loss'],
    tracking_uri=mlflow.get_tracking_uri(),
    log_model=True
)
# Log hyperparameters for this run.
mlflow_logger.log_hyperparams(CONFIG)

In [ ]:
# Set up Trainer
trainer = Trainer(
    callbacks=[
       EarlyStopping(
           monitor="val_loss",
           patience=3,
           verbose=True,
           mode="min",
       ),
    ],
    max_epochs=10,
    logger=mlflow_logger,
)
trainer.accelerator.name()

# Set up and Train Model
model = BESSTimeXer(**CONFIG, scaler=data.scaler)
trainer.fit(model, datamodule=data)

In [ ]:
# Evaluate trained Model
trainer.test(model, datamodule=data)